# Bulk Flight-Level Ingestion and Delay Labeling (OpenSky)

This notebook uses OpenSky flight endpoints only (`get_flights_from_interval`) to build a flight-level labeled dataset.

## Design notes
- We avoid `/states/all` and avoid paid schedule APIs.
- We query 2-hour windows because OpenSky interval calls should stay within API limits.
- The workflow is resumable: downloaded daily parquet files are skipped if present.
- Delay labels are route-relative, using `median_duration * 1.15` as threshold.

In [ ]:
import pandas as pd

from src.download_flights import download_flights
from src.build_route_baseline import build_route_baseline
from src.label_flights import label_flights
from src.qc_flights import generate_flight_qc


In [ ]:
# 1-week example window (change freely for bulk backfill)
START_DATE = "2022-01-01"
END_DATE = "2022-01-07"


In [ ]:
# 1) Download flights to daily parquet partitions
manifest_path = download_flights(start_date=START_DATE, end_date=END_DATE)
pd.read_csv(manifest_path).tail()


In [ ]:
# 2) Build route baseline from all downloaded flights
baseline_path = build_route_baseline()
pd.read_parquet(baseline_path).head()


In [ ]:
# 3) Label flights as late/on_time
labeled_path = label_flights()
labeled_df = pd.read_parquet(labeled_path)
labeled_df.head()


In [ ]:
# 4) Class distribution
labeled_df['delay_label'].value_counts(normalize=True).rename('fraction')


In [ ]:
# 5) Top 10 busiest routes
top_routes = (
    labeled_df.groupby(['estDepartureAirport', 'estArrivalAirport'])
    .size()
    .sort_values(ascending=False)
    .head(10)
)
top_routes


In [ ]:
# QC summary CSV required by the project
qc_path = generate_flight_qc(labeled_path=labeled_path)
pd.read_csv(qc_path)
